In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from benchmark import questions
from model_interface.qwen import Qwen

In [2]:
model = Qwen("models/qwen25-coder-7b")
model.load_model()
model.load_tokenizer()

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
system_prompt = """You are an expert in data integration. You are provided with:
- Question: A question that a data scientist wants to answer.
- Target Schema: A schema that the data scientist has determined to be sufficient to answer the question.
- Available Tables: A list of available tables in the database, each represented by its schema and a sample row.

Your task is to determine the precise steps required to construct a table that conforms to the Target Schema using the Available Tables.  
The operations include:
1. Select Table - Choose a single table
2. Table Joins - Combine data from multiple tables based on matching keys.
3. Row Filtering - Remove rows that do not satisfy a condition.
4. Column Selection - Choose relevant columns from the available tables.
5. Column Extraction - Extract information from one column to create a new column in the target schema.
  - Only use column extracts when strictly necessary (e.g., extracting "Country" from a "City" column).

Output Requirements
- The output must be a Python list of objects, where each object contains:
  - `"operation"`: The type of operation (`"join"`, `"filter"`, `"select"`, or `"extract"`).
  - `"description"`: A natural language description of what the operation does.
- Do not include any additional text, explanations, or formatting—only return a valid Python list that is directly parseable.

Example: Direct Match with a Join and Filter
Input:  
- Question: "List the names of all employees that handle transactions beyond 30000."  
- Target Schema: [`Employee ID`, `Employee Name`, `Department`]  
- Available Tables:  
  - Table1:  
    - Columns: `Emp_ID` | `Fullname` | `Dept`  
    - Sample row: `123` | `James Morgan` | `HR`  
  - Table2:  
    - Columns: `Transaction_ID` | `Emp_ID` | `Amount` | `Date`  
    - Sample row: `154` | `123` | `30000` | `2024-12-02`  

Output:
[
    {"operation": "join", "description": "Join Table1 and Table2 on column `Emp_ID` to associate employees with their transactions, resulting in Table3."},
    {"operation": "filter", "description": "Filter rows where `Amount` is greater than 30000 to include only high-value transactions."},
    {"operation": "select", "description": "Select columns `Emp_ID` (as Employee ID), `Fullname` (as Employee Name), and `Dept` (as Department) from the joined table."}
]
"""

In [4]:
import pandas as pd
from utils import format_schema
pneuma_retrieved_tables = ["pandas_dfs/codebase_community/comments.csv", "pandas_dfs/codebase_community/posts.csv", "pandas_dfs/california_schools/satscores.csv"]
schemas_samplerows: dict[str,str] = dict()
for idx, table in enumerate(pneuma_retrieved_tables):
    df = pd.read_csv(f"../TAG-Bench/{table}", nrows=1)
    schemas_samplerows[f"Table_{idx}"] = format_schema(df)
from ast import literal_eval
target_schema = literal_eval("['School ID', 'School Name', 'Math Score', 'Location']")
prompt = f"""Question: {questions[0]}
Target Schema: {target_schema}
Column to Fill: {target_schema[1]}

Available Tables:"""
for idx in schemas_samplerows.keys():
    table_content = schemas_samplerows[idx]
    prompt += f'\n\n{idx}:\n{table_content}'

In [5]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": prompt}
]

In [6]:
plan = model.chat(messages)
print(plan)

```python
[
    {"operation": "join", "description": "Join Table_2 with itself on the `sname` column to aggregate data for each school."},
    {"operation": "filter", "description": "Filter rows where `AvgScrMath` is greater than 560 to include only schools with an average math score above 560."},
    {"operation": "filter", "description": "Filter rows where `dname` is 'Bay Area' to include only schools located in the Bay Area."},
    {"operation": "select", "description": "Select columns `sname` (as School Name) from the filtered table."}
]
```
